# Benchmark: prepared one-arm WDM time-shift kernel

This notebook benchmarks the **current prepared one-arm plan** used by the
production **device-materialized** route.

The benchmark is deliberately conservative:

1. screening runs are used only to identify candidates;
2. no scheduling default is changed from a three-run screening result;
3. pairwise confirmation uses rotating execution order and reports the median,
   quartiles, and interquartile range (IQR);
4. a candidate is marked **adopt** only when its median improvement is large
   enough and the central 50% timing intervals do not overlap;
5. the job-block sweep always uses the committed production scheduling
   `(row_chunk_size, lag_block_size) = (32640, 17)` unless those constants are
   changed explicitly at the top of the notebook.

The notebook excludes TDI and the time-domain response. Those belong in
`getting_started.ipynb`. Here, the timed region is the warmed six-link one-arm
application after the two WDM transforms and after plan construction.

**Interpretation of timings**

- plan construction is reported separately;
- JAX compilation is excluded through explicit warm-up calls;
- every timed result is synchronized with `block_until_ready()`;
- the reference uses fixed `L_trunc = 25`, so scheduling sweeps do not change
  the time-shift approximation;
- CPU benchmark results should be collected on an otherwise idle machine.


In [1]:
# --- 1. Imports, environment, and source-file checks ---

import gc
import inspect
import os
import platform
import sys
import time
from datetime import datetime
from itertools import product

# ------------------------------------------------------------------
# Backend selection
#
# IMPORTANT:
# This must be set before importing jax, jax.numpy, FEW, WDM,
# WDM_LISAresponse, or anything else that may import JAX.
# ------------------------------------------------------------------

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    _normalise_one_arm_execution_mode,
)

import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM_LISAresponse.code.lisa_response.response_with_plan as response_with_plan_module
import WDM_LISAresponse
import WDM


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("Requested JAX platforms:", os.environ.get("JAX_PLATFORMS"))
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("one_arm_plan:", one_arm_plan_module.__file__)
print("response_with_plan:", response_with_plan_module.__file__)
print(
    "OneArmWDMPlan.apply_device:",
    inspect.signature(OneArmWDMPlan.apply_device),
)
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu", (
    "JAX did not initialize the CPU backend. Restart the kernel and "
    "run this cell first, before any JAX-related imports."
)

assert all(device.platform == "cpu" for device in jax.devices())

assert (
    _normalise_one_arm_execution_mode("materialized")
    == "materialized"
)
assert (
    "execution_mode"
    in inspect.signature(OneArmWDMPlan.apply_device).parameters
)

Timestamp: 2026-07-17T11:00:59
Python executable: /data/tk593/conda-envs/lisatools/bin/python
Python: 3.12.12
Platform: Linux-5.14.0-687.24.1.el9_8.x86_64-x86_64-with-glibc2.34
JAX: 0.9.2
Requested JAX platforms: cpu
JAX backend: cpu
JAX devices: [CpuDevice(id=0)]
WDM: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
one_arm_plan: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/code/lisa_response/one_arm_plan.py
response_with_plan: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/code/lisa_response/response_with_plan.py
OneArmWDMPlan.apply_device: (self, w_t_plus, w_t_cross, *, execution_mode: 'str' = 'materialized', return_profile: 'bool' = False, cache_device_plan: 'bool' = True)
RSS: 0.80 GiB


In [2]:
# --- 2. Scientific configuration and benchmark controls ---

# Full-year production-like case.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

# Windowing.
alpha = 0.005
tukey_factor = 4

# Orbit start and sky position.
ORBIT_FILE = 'esa-trailing-orbits.h5'
theta = np.pi / 2
phi = 0.0

# EMRI parameters.
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

# Fixed numerical settings. The sweep changes only scheduling parameters.
BASE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
}

# Current committed production schedule after analytic-parity promotion.
BASELINE_ROW_CHUNK = 32640
BASELINE_LAG_BLOCK = 17

# Targeted retuning for the analytic-parity kernel.
#
# L_trunc remains fixed at 25, giving 51 lags. In particular,
# lag_block_size=17 divides the lag count exactly:
#     51 = 3 * 17
ROW_CHUNK_SIZES = [
    6528,   # 5 exact chunks
    8160,   # 4 exact chunks
    10880,  # 3 exact chunks
    16320,  # 2 exact chunks
    32640, # 1 exact chunk
]
LAG_BLOCK_SIZES = [3, 17]

# Screening is used only to choose candidates for pairwise confirmation.
SCREEN_WARMUPS = 1
SCREEN_REPEATS = 3
TOP_K_CONFIRM = 4

# Pairwise confirmation.
CONFIRM_WARMUPS = 2
CONFIRM_REPEATS = 15

MIN_MEDIAN_IMPROVEMENT_PERCENT = 3.0
MAX_REL_L2_FOR_SCHEDULING = 1e-6

# Already tested and rejected for the production CPU route.
RUN_JOBBLOCK_SWEEP = False
JOBBLOCK_ROW_CHUNK = 4096
JOBBLOCK_LAG_BLOCK = 32
JOB_BLOCK_SIZES = [1, 2, 3, 6]
JOBBLOCK_WARMUPS = 2
JOBBLOCK_REPEATS = 15

SAVE_RESULTS = False
RESULTS_PREFIX = "one_arm_analytic_parity_schedule_sweep"

print('Full-year samples:', N)
print('Expected WDM Nt:', N // (2 * Nf))
print('L_trunc:', L_TRUNC, '| number of lags:', 2 * L_TRUNC + 1)
print('Screening combinations:', len(ROW_CHUNK_SIZES) * len(LAG_BLOCK_SIZES))
print(
    'Committed scheduling:',
    f'row={BASELINE_ROW_CHUNK}',
    f'lag_block={BASELINE_LAG_BLOCK}',
)
print(
    'Adoption guard:',
    f'median improvement >= {MIN_MEDIAN_IMPROVEMENT_PERCENT:.1f}%',
    'and non-overlapping IQRs',
)


Full-year samples: 3200000
Expected WDM Nt: 16000
L_trunc: 25 | number of lags: 51
Screening combinations: 10
Committed scheduling: row=32640 lag_block=17
Adoption guard: median improvement >= 3.0% and non-overlapping IQRs


In [3]:
# --- 3. Timing, validation, and display helpers ---

def block_result(value):
    # Synchronize all JAX leaves before stopping a timer.
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, 'block_until_ready'):
            leaf.block_until_ready()
    return value


def summarize_timings(timings):
    timings = np.asarray(timings, dtype=float)
    if timings.size == 0:
        raise ValueError('At least one timing is required.')

    q25_s, q75_s = np.percentile(timings, [25.0, 75.0])

    return {
        'median_s': float(np.median(timings)),
        'minimum_s': float(np.min(timings)),
        'maximum_s': float(np.max(timings)),
        'mean_s': float(np.mean(timings)),
        'std_s': float(np.std(timings)),
        'q25_s': float(q25_s),
        'q75_s': float(q75_s),
        'iqr_s': float(q75_s - q25_s),
        'times_s': [float(x) for x in timings],
    }


def benchmark_call(fn, *, warmups=1, repeats=3):
    # Warm, synchronize, and return all individual runtimes.
    warmups = max(0, int(warmups))
    repeats = max(1, int(repeats))

    result = None
    for _ in range(warmups):
        result = fn()
        block_result(result)

    timings = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        block_result(result)
        timings.append(time.perf_counter() - t0)

    return result, summarize_timings(timings)


def compare_device_batches(candidate, reference):
    # Compare two device-resident six-link batches using scalar reductions.
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    delta = candidate - reference
    max_abs_value = jnp.max(jnp.abs(delta))
    ref_norm = jnp.linalg.norm(reference)
    rel_l2_value = jnp.where(
        ref_norm > 0,
        jnp.linalg.norm(delta) / ref_norm,
        jnp.nan,
    )

    max_abs_value.block_until_ready()
    rel_l2_value.block_until_ready()

    return {
        'max_abs': float(max_abs_value),
        'rel_l2': float(rel_l2_value),
    }


def assess_paired_candidate(
    baseline_times,
    candidate_times,
    *,
    rel_l2,
    min_improvement_percent=MIN_MEDIAN_IMPROVEMENT_PERCENT,
    max_rel_l2=MAX_REL_L2_FOR_SCHEDULING,
):
    baseline = summarize_timings(baseline_times)
    candidate = summarize_timings(candidate_times)

    runtime_change_percent = (
        100.0 * (candidate['median_s'] / baseline['median_s'] - 1.0)
    )
    improvement_percent = -runtime_change_percent
    numerically_valid = bool(np.isfinite(rel_l2) and rel_l2 <= max_rel_l2)

    # Conservative distributional guard:
    # at least 75% of candidate timings must be below at least 75% of baseline
    # timings. Equivalently, the central 50% intervals do not overlap.
    iqr_separated = bool(candidate['q75_s'] < baseline['q25_s'])

    if (
        numerically_valid
        and improvement_percent >= min_improvement_percent
        and iqr_separated
    ):
        decision = 'adopt'
    elif numerically_valid and improvement_percent >= min_improvement_percent:
        decision = 'retest'
    else:
        decision = 'retain_baseline'

    return {
        'baseline': baseline,
        'candidate': candidate,
        'runtime_change_percent': float(runtime_change_percent),
        'improvement_percent': float(improvement_percent),
        'numerically_valid': numerically_valid,
        'iqr_separated': iqr_separated,
        'decision': decision,
    }


def make_shift_kwargs(
    row_chunk_size,
    lag_block_size,
    *,
    backend='lagfirst_chunked_lagblock',
    job_block_size=1,
):
    kwargs = dict(BASE_SHIFT_KWARGS)
    kwargs.update({
        'assembly_backend': str(backend),
        'row_chunk_size': int(row_chunk_size),
        'lag_block_size': int(lag_block_size),
        'job_block_size': int(job_block_size),
    })
    return kwargs


def build_one_arm_plan(grid_context, kwargs):
    t0 = time.perf_counter()
    plan = OneArmWDMPlan.build_from_context(
        grid_context,
        orbits_trailing,
        theta,
        phi,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=kwargs,
        use_jax=True,
    )
    build_s = time.perf_counter() - t0
    return plan, float(build_s)


def show_rows(rows, *, sort_by=None):
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by is not None and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, kind='stable').reset_index(drop=True)
        display(frame)
        return frame

    rows_out = list(rows)
    if sort_by is not None:
        rows_out = sorted(rows_out, key=lambda row: row.get(sort_by, np.inf))
    for row in rows_out:
        print(row)
    return rows_out


def save_rows(rows, filename):
    if not SAVE_RESULTS:
        return
    if pd is None:
        print('pandas is unavailable; skipping CSV export:', filename)
        return
    pd.DataFrame(rows).to_csv(filename, index=False)
    print('Saved:', filename)


## 4. Generate the fixed waveform and orbit object

This is identical in spirit to the production getting-started case. The same
waveform is reused for every candidate. Its generation time is not part of the
shift benchmark.


In [ ]:
# --- 4a. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    'td_gen': td_gen,
    'm1': m1,
    'm2': m2,
    'a': a,
    'p0': p0,
    'e0': e0,
    'x0': x0,
    'dist': dist,
    'theta': theta,
    'phi': phi,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0,
    'eps': eps,
}

window_kwargs = {
    'alpha': alpha,
    'tukey_factor': tukey_factor,
}

print('Orbit samples:', orbit_data.size)
print('Starting waveform generation...')


Orbit samples: 1721
Starting waveform generation...


In [ ]:
# --- 4b. Generate and pad the waveform ---

t_wave0 = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model='emri',
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function='tukey',
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

waveform_generation_s = time.perf_counter() - t_wave0

print('Waveform generation_s:', f'{waveform_generation_s:.3f}')
print('times_pad:', times_pad.shape, times_pad.dtype)
print('hplus_td:', hplus_td.shape, hplus_td.dtype)
print('hcross_td:', hcross_td.shape, hcross_td.dtype)
print('WDM Nt x Nf:', pad_info['wdm_Nt'], 'x', Nf)
print('Tail right padding:', pad_info['wdm_tail_pad_right'])
print('RSS:', f'{rss_gib():.2f} GiB')


Waveform generation_s: 8.846
times_pad: (3264000,) float64
hplus_td: (3264000,) float64
hcross_td: (3264000,) float64
WDM Nt x Nf: 32640 x 100
Tail right padding: 0
RSS: 6.50 GiB


## 5. Build the WDM grid and perform the two DWTs once

The WDM grid and waveform coefficients are shared by all candidates. The
reported candidate timings therefore isolate the prepared six-link
materialized one-arm calculation.


In [ ]:
# --- 5. Shared WDM grid and waveform context ---

t_grid0 = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - t_grid0

t_dwt0 = time.perf_counter()
wave_context = _build_wdm_dT_context_from_wdm(
    times_pad,
    hplus_td,
    hcross_td,
    grid_context['wdm'],
)
block_result((wave_context['w_t_plus'], wave_context['w_t_cross']))
two_dwt_s = time.perf_counter() - t_dwt0

wdm = grid_context['wdm']
w_plus_device = jnp.asarray(wave_context['w_t_plus'], dtype=jnp.complex64)
w_cross_device = jnp.asarray(wave_context['w_t_cross'], dtype=jnp.complex64)
block_result((w_plus_device, w_cross_device))

print('WDM grid build_s:', f'{grid_build_s:.3f}')
print('Two DWTs + synchronization_s:', f'{two_dwt_s:.3f}')
print('WDM Nt:', wdm.Nt)
print('WDM Nf:', wdm.Nf)
print('WDM N:', wdm.N)
print('WDM dT:', wdm.dT)
print('WDM dF:', wdm.dF)
print('plus WDM:', w_plus_device.shape, w_plus_device.dtype)
print('cross WDM:', w_cross_device.shape, w_cross_device.dtype)
print('RSS:', f'{rss_gib():.2f} GiB')


WDM grid build_s: 2.870
Two DWTs + synchronization_s: 0.767
WDM Nt: 32640
WDM Nf: 100
WDM N: 3264000
WDM dT: 1000.0
WDM dF: 0.0005
plus WDM: (32640, 100) complex64
cross WDM: (32640, 100) complex64
RSS: 7.30 GiB


## 6. Establish the committed production baseline

The committed analytic-parity production reference is:

```python
row_chunk_size = 32640
lag_block_size = 17
assembly_backend = "lagfirst_chunked_lagblock"
assembly_precision = "complex64"
L_trunc = 25
```

The reference output remains on-device. Candidate validation uses only scalar
device reductions, so it does not repeatedly copy the full six-link arrays to
NumPy.


In [ ]:
# --- 6. Baseline plan, reference output, and warmed timing ---

baseline_kwargs = make_shift_kwargs(
    BASELINE_ROW_CHUNK,
    BASELINE_LAG_BLOCK,
)

baseline_plan, baseline_plan_build_s = build_one_arm_plan(
    grid_context,
    baseline_kwargs,
)

assert (
    baseline_plan.shift_plan.resolved_assembly_backend
    == 'lagfirst_chunked_lagblock'
)

def run_baseline():
    return baseline_plan.apply_batch_device_materialized(
        w_plus_device,
        w_cross_device,
    )

reference_device, baseline_timing = benchmark_call(
    run_baseline,
    warmups=1,
    repeats=CONFIRM_REPEATS,
)

print('Baseline kwargs:', baseline_kwargs)
print('Baseline plan build_s:', f'{baseline_plan_build_s:.3f}')
print('Baseline warmed median_s:', f"{baseline_timing['median_s']:.3f}")
print('Baseline minimum_s:', f"{baseline_timing['minimum_s']:.3f}")
print('Baseline maximum_s:', f"{baseline_timing['maximum_s']:.3f}")
print('Baseline q25_s:', f"{baseline_timing['q25_s']:.3f}")
print('Baseline q75_s:', f"{baseline_timing['q75_s']:.3f}")
print('Baseline IQR_s:', f"{baseline_timing['iqr_s']:.3f}")
print('Baseline all:', [round(x, 3) for x in baseline_timing['times_s']])
print('Reference shape:', reference_device.shape)
print('Reference dtype:', reference_device.dtype)
print(
    'Persistent kernel memory_MiB:',
    baseline_plan.shift_plan.kernel_memory_bytes / 1024**2,
)
print('RSS:', f'{rss_gib():.2f} GiB')


Baseline kwargs: {'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear', 'use_jax': True, 'row_chunk_size': 32640, 'lag_block_size': 17, 'job_block_size': 1}
Baseline plan build_s: 1.029
Baseline warmed median_s: 0.429
Baseline minimum_s: 0.403
Baseline maximum_s: 0.456
Baseline q25_s: 0.423
Baseline q75_s: 0.438
Baseline IQR_s: 0.014
Baseline all: [0.427, 0.403, 0.439, 0.428, 0.411, 0.448, 0.456, 0.426, 0.437, 0.42, 0.431, 0.44, 0.43, 0.429, 0.419]
Reference shape: (6, 32640, 100)
Reference dtype: complex64
Persistent kernel memory_MiB: 329.70703125
RSS: 8.59 GiB


## 7. Screen row-chunk and lag-block combinations

This is a **screening-only** sweep. It keeps the lag support and transfer-kernel
interpolation fixed. Every candidate is warmed once and timed three times by
default.

The table reports the median and IQR, but no production setting is changed from
this cell. The fastest candidates merely advance to the longer rotating-order
pairwise confirmation.


In [ ]:
# --- 7. Initial scheduling sweep ---

screen_rows = []

for row_chunk_size, lag_block_size in product(
    ROW_CHUNK_SIZES,
    LAG_BLOCK_SIZES,
):
    kwargs = make_shift_kwargs(row_chunk_size, lag_block_size)

    is_baseline = (
        row_chunk_size == BASELINE_ROW_CHUNK
        and lag_block_size == BASELINE_LAG_BLOCK
    )

    print(
        '\nCandidate:',
        f'row={row_chunk_size}',
        f'lag_block={lag_block_size}',
        '(baseline)' if is_baseline else '',
    )

    if is_baseline:
        plan = baseline_plan
        plan_build_s = baseline_plan_build_s
    else:
        plan, plan_build_s = build_one_arm_plan(grid_context, kwargs)

    def run_candidate(plan=plan):
        return plan.apply_batch_device_materialized(
            w_plus_device,
            w_cross_device,
        )

    candidate_device, timing = benchmark_call(
        run_candidate,
        warmups=SCREEN_WARMUPS,
        repeats=SCREEN_REPEATS,
    )
    comparison = compare_device_batches(candidate_device, reference_device)

    row = {
        'backend': plan.shift_plan.resolved_assembly_backend,
        'row_chunk_size': int(row_chunk_size),
        'lag_block_size': int(lag_block_size),
        'job_block_size': int(
            getattr(plan.shift_plan.config, 'job_block_size', 1)
        ),
        'plan_build_s': float(plan_build_s),
        **timing,
        **comparison,
        'speedup_vs_baseline': (
            baseline_timing['median_s'] / timing['median_s']
        ),
        'kernel_memory_mib': (
            plan.shift_plan.kernel_memory_bytes / 1024**2
        ),
        'screening_only': True,
    }
    screen_rows.append(row)

    print(
        f"median={timing['median_s']:.3f} s | "
        f"IQR={timing['iqr_s']:.3f} s | "
        f"speedup={row['speedup_vs_baseline']:.3f}x | "
        f"rel_l2={comparison['rel_l2']:.3e}"
    )

    if not is_baseline:
        del candidate_device
        del plan
        gc.collect()

screen_table = show_rows(screen_rows, sort_by='median_s')
#save_rows(screen_rows, f'{RESULTS_PREFIX}_screen.csv')



Candidate: row=6528 lag_block=3 
median=1.261 s | IQR=0.017 s | speedup=0.340x | rel_l2=1.214e-07

Candidate: row=6528 lag_block=17 
median=0.511 s | IQR=0.004 s | speedup=0.838x | rel_l2=0.000e+00

Candidate: row=8160 lag_block=3 
median=0.460 s | IQR=0.003 s | speedup=0.933x | rel_l2=1.214e-07

Candidate: row=8160 lag_block=17 
median=0.492 s | IQR=0.005 s | speedup=0.871x | rel_l2=0.000e+00

Candidate: row=10880 lag_block=3 
median=0.627 s | IQR=0.006 s | speedup=0.683x | rel_l2=1.214e-07

Candidate: row=10880 lag_block=17 
median=0.472 s | IQR=0.005 s | speedup=0.908x | rel_l2=0.000e+00

Candidate: row=16320 lag_block=3 
median=0.607 s | IQR=0.005 s | speedup=0.706x | rel_l2=1.214e-07

Candidate: row=16320 lag_block=17 
median=0.454 s | IQR=0.008 s | speedup=0.945x | rel_l2=0.000e+00

Candidate: row=32640 lag_block=3 
median=0.511 s | IQR=0.004 s | speedup=0.839x | rel_l2=1.214e-07

Candidate: row=32640 lag_block=17 (baseline)
median=0.433 s | IQR=0.003 s | speedup=0.989x | rel_l2

,backend,row_chunk_size,lag_block_size,job_block_size,plan_build_s,median_s,minimum_s,maximum_s,mean_s,std_s,q25_s,q75_s,iqr_s,times_s,max_abs,rel_l2,speedup_vs_baseline,kernel_memory_mib,screening_only
0,lagfirst_chunked_lagblock,32640,17,1,1.028702,0.433376,0.431531,0.436952,0.433953,0.002250,0.432454,0.435164,0.002711,"[0.4333762879832648, 0.4369518969906494, 0.431...",0.000000e+00,0.000000e+00,0.988922,329.707031,True
1,lagfirst_chunked_lagblock,16320,17,1,0.638515,0.453621,0.440727,0.456612,0.450320,0.006892,0.447174,0.455116,0.007942,"[0.45661188702797517, 0.4536206590128131, 0.44...",0.000000e+00,0.000000e+00,0.944788,329.707031,True
2,lagfirst_chunked_lagblock,8160,3,1,0.642890,0.459561,0.454385,0.460248,0.458065,0.002617,0.456973,0.459905,0.002932,"[0.45438483502948657, 0.4595611320110038, 0.46...",2.168404e-18,1.213612e-07,0.932575,329.707031,True
3,lagfirst_chunked_lagblock,10880,17,1,0.643818,0.472051,0.469380,0.480159,0.473864,0.004583,0.470716,0.476105,0.005390,"[0.47205100499559194, 0.48015930998371914, 0.4...",0.000000e+00,0.000000e+00,0.907900,329.707031,True
4,lagfirst_chunked_lagblock,8160,17,1,0.643484,0.491998,0.491843,0.502288,0.495376,0.004888,0.491920,0.497143,0.005223,"[0.49184256099397317, 0.49199762602802366, 0.5...",0.000000e+00,0.000000e+00,0.871092,329.707031,True
5,lagfirst_chunked_lagblock,32640,3,1,0.652237,0.511065,0.508546,0.515920,0.511844,0.003060,0.509806,0.513493,0.003687,"[0.5110654219752178, 0.5159201640053652, 0.508...",2.168404e-18,1.213612e-07,0.838592,329.707031,True
6,lagfirst_chunked_lagblock,6528,17,1,0.641755,0.511129,0.508134,0.515839,0.511701,0.003171,0.509632,0.513484,0.003852,"[0.5158389839925803, 0.511129325022921, 0.5081...",0.000000e+00,0.000000e+00,0.838487,329.707031,True
7,lagfirst_chunked_lagblock,16320,3,1,0.637294,0.607376,0.599602,0.609177,0.605385,0.004154,0.603489,0.608276,0.004787,"[0.6073761510197073, 0.5996024569612928, 0.609...",2.168404e-18,1.213612e-07,0.705617,329.707031,True
8,lagfirst_chunked_lagblock,10880,3,1,0.652562,0.627270,0.621384,0.633088,0.627247,0.004778,0.624327,0.630179,0.005852,"[0.6272697239764966, 0.6330880390014499, 0.621...",2.168404e-18,1.213612e-07,0.683239,329.707031,True
9,lagfirst_chunked_lagblock,6528,3,1,0.651914,1.260589,1.237527,1.272118,1.256745,0.014381,1.249058,1.266354,0.017295,"[1.2605893820291385, 1.2721180110238492, 1.237...",2.168404e-18,1.213612e-07,0.339980,329.707031,True


## 8. Pairwise confirmation with an IQR adoption guard

The best screening candidates are confirmed one at a time against the committed
baseline. Both routes receive explicit warm-ups and their execution order
alternates.

A candidate is marked:

- **adopt** only when the median improves by at least the configured threshold,
  the numerical error is acceptable, and the candidate and baseline IQRs do not
  overlap;
- **retest** when the median improvement is large enough but the IQRs overlap;
- **retain_baseline** otherwise.

This conservative rule prevents a noisy three-run screen or a single low median
from silently changing the scheduling used by later experiments.


In [ ]:
# --- 8. Confirm the best candidates against the baseline ---

ranked_screen = sorted(screen_rows, key=lambda row: row['median_s'])

candidate_keys = []
for row in ranked_screen:
    key = (int(row['row_chunk_size']), int(row['lag_block_size']))
    if key == (BASELINE_ROW_CHUNK, BASELINE_LAG_BLOCK):
        continue
    if key not in candidate_keys:
        candidate_keys.append(key)
    if len(candidate_keys) >= TOP_K_CONFIRM:
        break

print('Candidates selected for pairwise confirmation:', candidate_keys)

confirm_rows = []

for row_chunk_size, lag_block_size in candidate_keys:
    kwargs = make_shift_kwargs(row_chunk_size, lag_block_size)
    candidate_plan, candidate_plan_build_s = build_one_arm_plan(
        grid_context,
        kwargs,
    )

    def run_candidate(candidate_plan=candidate_plan):
        return candidate_plan.apply_batch_device_materialized(
            w_plus_device,
            w_cross_device,
        )

    # Compile and warm both routes before timing.
    for _ in range(CONFIRM_WARMUPS):
        block_result(run_baseline())
        block_result(run_candidate())

    candidate_output = run_candidate()
    block_result(candidate_output)
    comparison = compare_device_batches(candidate_output, reference_device)

    baseline_pair_times = []
    candidate_pair_times = []

    for repeat_index in range(CONFIRM_REPEATS):
        if repeat_index % 2 == 0:
            order = (('baseline', run_baseline), ('candidate', run_candidate))
        else:
            order = (('candidate', run_candidate), ('baseline', run_baseline))

        for label, fn in order:
            t0 = time.perf_counter()
            output = fn()
            block_result(output)
            elapsed = time.perf_counter() - t0

            if label == 'baseline':
                baseline_pair_times.append(float(elapsed))
            else:
                candidate_pair_times.append(float(elapsed))

    assessment = assess_paired_candidate(
        baseline_pair_times,
        candidate_pair_times,
        rel_l2=comparison['rel_l2'],
    )
    baseline_stats = assessment['baseline']
    candidate_stats = assessment['candidate']

    row = {
        'row_chunk_size': int(row_chunk_size),
        'lag_block_size': int(lag_block_size),
        'candidate_plan_build_s': float(candidate_plan_build_s),
        'baseline_pair_median_s': baseline_stats['median_s'],
        'baseline_q25_s': baseline_stats['q25_s'],
        'baseline_q75_s': baseline_stats['q75_s'],
        'baseline_iqr_s': baseline_stats['iqr_s'],
        'candidate_median_s': candidate_stats['median_s'],
        'candidate_minimum_s': candidate_stats['minimum_s'],
        'candidate_maximum_s': candidate_stats['maximum_s'],
        'candidate_q25_s': candidate_stats['q25_s'],
        'candidate_q75_s': candidate_stats['q75_s'],
        'candidate_iqr_s': candidate_stats['iqr_s'],
        'speedup_vs_paired_baseline': (
            baseline_stats['median_s'] / candidate_stats['median_s']
        ),
        'runtime_change_percent': assessment['runtime_change_percent'],
        'improvement_percent': assessment['improvement_percent'],
        'iqr_separated': assessment['iqr_separated'],
        'numerically_valid': assessment['numerically_valid'],
        'decision': assessment['decision'],
        'baseline_times_s': baseline_stats['times_s'],
        'candidate_times_s': candidate_stats['times_s'],
        **comparison,
    }
    confirm_rows.append(row)

    print(
        '\nConfirmed:',
        f'row={row_chunk_size}',
        f'lag_block={lag_block_size}',
    )
    print(
        '  baseline:',
        f"median={baseline_stats['median_s']:.3f} s | "
        f"IQR={baseline_stats['iqr_s']:.3f} s | "
        f"[q25,q75]=[{baseline_stats['q25_s']:.3f}, "
        f"{baseline_stats['q75_s']:.3f}]"
    )
    print(
        '  candidate:',
        f"median={candidate_stats['median_s']:.3f} s | "
        f"IQR={candidate_stats['iqr_s']:.3f} s | "
        f"[q25,q75]=[{candidate_stats['q25_s']:.3f}, "
        f"{candidate_stats['q75_s']:.3f}]"
    )
    print(
        '  runtime change:',
        f"{assessment['runtime_change_percent']:+.2f}%"
    )
    print('  IQR separated:', assessment['iqr_separated'])
    print('  decision:', assessment['decision'])
    print('  rel_l2:', f"{comparison['rel_l2']:.3e}")
    print(
        '  candidate all:',
        [round(x, 3) for x in candidate_stats['times_s']],
    )

    del candidate_output
    del candidate_plan
    gc.collect()

confirm_table = show_rows(confirm_rows, sort_by='candidate_median_s')
#save_rows(confirm_rows, f'{RESULTS_PREFIX}_confirm.csv')

adopted_confirmed = [
    row for row in confirm_rows
    if row['decision'] == 'adopt'
]

if adopted_confirmed:
    best_adopted = min(
        adopted_confirmed,
        key=lambda row: row['candidate_median_s'],
    )
    print(
        '\nStrongly confirmed scheduling candidate:',
        (
            int(best_adopted['row_chunk_size']),
            int(best_adopted['lag_block_size']),
        ),
    )
else:
    print(
        '\nNo candidate passed the conservative adoption guard. '
        'Retain the committed scheduling:',
        (BASELINE_ROW_CHUNK, BASELINE_LAG_BLOCK),
    )


Candidates selected for pairwise confirmation: [(16320, 17), (8160, 3), (10880, 17), (8160, 17)]

Confirmed: row=16320 lag_block=17
  baseline: median=0.426 s | IQR=0.011 s | [q25,q75]=[0.424, 0.434]
  candidate: median=0.453 s | IQR=0.011 s | [q25,q75]=[0.449, 0.460]
  runtime change: +6.12%
  IQR separated: False
  decision: retain_baseline
  rel_l2: 0.000e+00
  candidate all: [0.459, 0.446, 0.478, 0.463, 0.453, 0.449, 0.449, 0.465, 0.447, 0.451, 0.451, 0.448, 0.458, 0.462, 0.453]

Confirmed: row=8160 lag_block=3
  baseline: median=0.431 s | IQR=0.013 s | [q25,q75]=[0.423, 0.436]
  candidate: median=0.527 s | IQR=0.057 s | [q25,q75]=[0.501, 0.558]
  runtime change: +22.17%
  IQR separated: False
  decision: retain_baseline
  rel_l2: 1.214e-07
  candidate all: [0.588, 0.54, 0.58, 0.488, 0.577, 0.485, 0.495, 0.495, 0.604, 0.51, 0.527, 0.539, 0.508, 0.534, 0.524]

Confirmed: row=10880 lag_block=17
  baseline: median=0.520 s | IQR=0.031 s | [q25,q75]=[0.492, 0.522]
  candidate: median=0.

,row_chunk_size,lag_block_size,candidate_plan_build_s,baseline_pair_median_s,baseline_q25_s,baseline_q75_s,baseline_iqr_s,candidate_median_s,candidate_minimum_s,candidate_maximum_s,...,speedup_vs_paired_baseline,runtime_change_percent,improvement_percent,iqr_separated,numerically_valid,decision,baseline_times_s,candidate_times_s,max_abs,rel_l2
0,16320,17,0.641750,0.426414,0.423592,0.434118,0.010526,0.452509,0.445694,0.478371,...,0.942333,6.119580,-6.119580,False,True,retain_baseline,"[0.42491446202620864, 0.4235460519557819, 0.41...","[0.45871966902632266, 0.44569419097388163, 0.4...",0.000000e+00,0.000000e+00
1,8160,17,0.647214,0.429634,0.424221,0.440095,0.015874,0.497202,0.475190,0.512986,...,0.864104,15.726789,-15.726789,False,True,retain_baseline,"[0.418455642007757, 0.44258774199988693, 0.429...","[0.4918210420291871, 0.4751901510171592, 0.493...",0.000000e+00,0.000000e+00
2,8160,3,0.640765,0.431138,0.422761,0.435636,0.012875,0.526713,0.485386,0.603566,...,0.818544,22.168070,-22.168070,False,True,retain_baseline,"[0.4112979039782658, 0.4303187580080703, 0.426...","[0.5881708479719236, 0.5395869580097497, 0.580...",2.168404e-18,1.213612e-07
3,10880,17,0.641410,0.519552,0.491682,0.522203,0.030521,0.560693,0.494001,0.569679,...,0.926624,7.918631,-7.918631,False,True,retain_baseline,"[0.4390761879622005, 0.4333923120284453, 0.523...","[0.4945893220137805, 0.4940013120067306, 0.560...",0.000000e+00,0.000000e+00



No candidate passed the conservative adoption guard. Retain the committed scheduling: (32640, 17)


## 9. Optional job-block sweep at fixed production scheduling

The job-block backend is compared **pairwise** against the ordinary production
backend using the explicit fixed scheduling:

```python
row_chunk_size = JOBBLOCK_ROW_CHUNK
lag_block_size = JOBBLOCK_LAG_BLOCK
```

It does not inherit the winner of the row/lag screening or confirmation cells.
This prevents a noisy earlier result from changing two experimental variables at
once.

Each job-block size is warmed, then timed in alternating order against the
ordinary backend. Median, quartiles, IQR, numerical agreement, and the same
conservative adoption decision are reported.


In [ ]:
# --- 9. Optional job-block scheduling sweep ---

jobblock_rows = []

if RUN_JOBBLOCK_SWEEP:
    jobblock_row_chunk = int(JOBBLOCK_ROW_CHUNK)
    jobblock_lag_block = int(JOBBLOCK_LAG_BLOCK)


    print(
        'Job-block base scheduling:',
        jobblock_row_chunk,
        jobblock_lag_block,
    )
    print(
        'Ordinary comparison backend:',
        baseline_plan.shift_plan.resolved_assembly_backend,
    )

    for job_block_size in JOB_BLOCK_SIZES:
        kwargs = make_shift_kwargs(
            jobblock_row_chunk,
            jobblock_lag_block,
            backend='lagfirst_chunked_lagblock_jobblock',
            job_block_size=job_block_size,
        )

        plan, plan_build_s = build_one_arm_plan(grid_context, kwargs)

        def run_jobblock(plan=plan):
            return plan.apply_batch_device_materialized(
                w_plus_device,
                w_cross_device,
            )

        # Compile and warm both routes.
        for _ in range(JOBBLOCK_WARMUPS):
            block_result(run_baseline())
            block_result(run_jobblock())

        output = run_jobblock()
        block_result(output)
        comparison = compare_device_batches(output, reference_device)

        ordinary_times = []
        jobblock_times = []

        for repeat_index in range(JOBBLOCK_REPEATS):
            if repeat_index % 2 == 0:
                order = (
                    ('ordinary', run_baseline),
                    ('jobblock', run_jobblock),
                )
            else:
                order = (
                    ('jobblock', run_jobblock),
                    ('ordinary', run_baseline),
                )

            for label, fn in order:
                t0 = time.perf_counter()
                timed_output = fn()
                block_result(timed_output)
                elapsed = time.perf_counter() - t0

                if label == 'ordinary':
                    ordinary_times.append(float(elapsed))
                else:
                    jobblock_times.append(float(elapsed))

        assessment = assess_paired_candidate(
            ordinary_times,
            jobblock_times,
            rel_l2=comparison['rel_l2'],
        )
        ordinary_stats = assessment['baseline']
        jobblock_stats = assessment['candidate']

        row = {
            'backend': plan.shift_plan.resolved_assembly_backend,
            'row_chunk_size': jobblock_row_chunk,
            'lag_block_size': jobblock_lag_block,
            'job_block_size': int(job_block_size),
            'plan_build_s': float(plan_build_s),
            'ordinary_median_s': ordinary_stats['median_s'],
            'ordinary_q25_s': ordinary_stats['q25_s'],
            'ordinary_q75_s': ordinary_stats['q75_s'],
            'ordinary_iqr_s': ordinary_stats['iqr_s'],
            'jobblock_median_s': jobblock_stats['median_s'],
            'jobblock_minimum_s': jobblock_stats['minimum_s'],
            'jobblock_maximum_s': jobblock_stats['maximum_s'],
            'jobblock_q25_s': jobblock_stats['q25_s'],
            'jobblock_q75_s': jobblock_stats['q75_s'],
            'jobblock_iqr_s': jobblock_stats['iqr_s'],
            'speedup_vs_ordinary': (
                ordinary_stats['median_s'] / jobblock_stats['median_s']
            ),
            'runtime_change_percent': assessment['runtime_change_percent'],
            'improvement_percent': assessment['improvement_percent'],
            'iqr_separated': assessment['iqr_separated'],
            'numerically_valid': assessment['numerically_valid'],
            'decision': assessment['decision'],
            'ordinary_times_s': ordinary_stats['times_s'],
            'jobblock_times_s': jobblock_stats['times_s'],
            **comparison,
        }
        jobblock_rows.append(row)

        print(
            f'\njob_block={job_block_size}'
        )
        print(
            '  ordinary:',
            f"median={ordinary_stats['median_s']:.3f} s | "
            f"IQR={ordinary_stats['iqr_s']:.3f} s | "
            f"[q25,q75]=[{ordinary_stats['q25_s']:.3f}, "
            f"{ordinary_stats['q75_s']:.3f}]"
        )
        print(
            '  job-block:',
            f"median={jobblock_stats['median_s']:.3f} s | "
            f"IQR={jobblock_stats['iqr_s']:.3f} s | "
            f"[q25,q75]=[{jobblock_stats['q25_s']:.3f}, "
            f"{jobblock_stats['q75_s']:.3f}]"
        )
        print(
            '  runtime change:',
            f"{assessment['runtime_change_percent']:+.2f}%"
        )
        print('  IQR separated:', assessment['iqr_separated'])
        print('  decision:', assessment['decision'])
        print('  rel_l2:', f"{comparison['rel_l2']:.3e}")
        print(
            '  all:',
            [round(x, 3) for x in jobblock_stats['times_s']],
        )

        del output
        del plan
        gc.collect()

    jobblock_table = show_rows(
        jobblock_rows,
        sort_by='jobblock_median_s',
    )
    #save_rows(jobblock_rows, f'{RESULTS_PREFIX}_jobblock.csv')

    adopted_jobblocks = [
        row for row in jobblock_rows
        if row['decision'] == 'adopt'
    ]

    if adopted_jobblocks:
        best_jobblock = min(
            adopted_jobblocks,
            key=lambda row: row['jobblock_median_s'],
        )
        print(
            '\nStrongly confirmed job-block candidate:',
            f"job_block_size={best_jobblock['job_block_size']}",
        )
    else:
        print(
            '\nNo job-block candidate passed the conservative adoption guard. '
            'Retain the ordinary production backend.'
        )
else:
    print('RUN_JOBBLOCK_SWEEP is False; skipping job-block tests.')


RUN_JOBBLOCK_SWEEP is False; skipping job-block tests.


## 10. Benchmark discipline for later kernel changes

For every structural experiment:

1. restart the kernel so the imported source is unambiguous;
2. run on an otherwise idle machine;
3. keep the committed reference output and scheduling fixed;
4. warm both routes before timing;
5. alternate execution order;
6. report all timings, median, quartiles, and IQR;
7. require acceptable complex64-level numerical agreement;
8. treat an apparent median improvement with overlapping IQRs as **retest**,
   not as a production switch;
9. change only one algorithmic feature at a time;
10. rerun `getting_started.ipynb` only after a candidate has passed the
    dedicated benchmark and is integrated into the production route.

The next structural candidates should be tested separately:

- generate phases by recurrence instead of a full exponential grid;
- split interior rows from boundary rows;
- use a contiguous materialized-source halo;
- simplify the checkerboard `Cnm` algebra.
